In [1]:
import pandas as pd
from pyproj import Transformer
import numpy as np
import os

# ==========================================
# 1. Configuration and Constants
# ==========================================
BASE_PATH = '..' 
CRS_SOURCE = "EPSG:4326"   # WGS84
CRS_TARGET = "EPSG:31984"  # SIRGAS 2000 / UTM zone 24S

def main():
    print("--- Starting Robust Coordinate Conversion ---")

    # Path configuration
    input_csv_path = os.path.join(BASE_PATH, 'includes', 'Tabela_consumidores_Itapua.csv')
    output_csv_path = os.path.join(BASE_PATH, 'includes', 'Tabela_consumidores_Itapua_convertida.csv')

    if not os.path.exists(input_csv_path):
        print(f"Error: Input file not found at {input_csv_path}")
        return

    # Load data
    print(f"Reading file: {input_csv_path}")
    df = pd.read_csv(input_csv_path, delimiter=";")
    original_count = len(df)

    # ==========================================
    # 2. Data Pre-processing
    # ==========================================
    # Replace NaN coordinates with 0 to prevent Transformer crash
    # These will result in (0,0) or invalid UTM values which we handle later
    df["LONG_GEO"] = df["LONG_GEO"].fillna(0)
    df["LAT_GEO"] = df["LAT_GEO"].fillna(0)

    # ==========================================
    # 3. Coordinate Transformation
    # ==========================================
    print(f"Transforming coordinates from {CRS_SOURCE} to {CRS_TARGET}...")

    # Initialize Transformer (always_xy=True ensures Longitude/Latitude order)
    transformer = Transformer.from_crs(CRS_SOURCE, CRS_TARGET, always_xy=True)

    # Perform vectorized transformation
    try:
        x_values, y_values = transformer.transform(
            df["LONG_GEO"].values, 
            df["LAT_GEO"].values
        )

        # Handle invalid projection results (Inf or NaN)
        # UTM zone 24S cannot project (0,0) Latitude/Longitude correctly,
        # so we force these to 0.0 to keep the rows alive in GAMA.
        df["X"] = np.where(np.isinf(x_values) | np.isnan(x_values), 0.0, x_values)
        df["Y"] = np.where(np.isinf(y_values) | np.isnan(y_values), 0.0, y_values)

    except Exception as e:
        print(f"An error occurred during transformation: {e}")
        return

    # ==========================================
    # 4. Save and Report
    # ==========================================
    # IMPORTANT: We DO NOT use dropna() here to preserve all records.
    # Agents with X=0.0 and Y=0.0 will still compute consumption in GAMA.
    df.to_csv(output_csv_path, sep=";", index=False)

    failed_coords = len(df[df["X"] == 0.0])

    print("-" * 40)
    print(f"Processing Complete.")
    print(f"Original records: {original_count}")
    print(f"Saved records: {len(df)}")
    print(f"Records with forced (0,0) coordinates: {failed_coords}")
    print(f"Output saved to: {output_csv_path}")
    print("-" * 40)

if __name__ == "__main__":
    main()

--- Starting Robust Coordinate Conversion ---
Reading file: ..\includes\Tabela_consumidores_Itapua.csv
Transforming coordinates from EPSG:4326 to EPSG:31984...


----------------------------------------
Processing Complete.
Original records: 19630
Saved records: 19630
Records with forced (0,0) coordinates: 0
Output saved to: ..\includes\Tabela_consumidores_Itapua_convertida.csv
----------------------------------------
